In [ ]:
import sys
import os
import copy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
sys.path.append(os.path.join(os.path.pardir, 'lesview'))
sys.path.append(os.path.join(os.path.pardir, 'gotmtool'))
from gotmtool import dat_dump_pfl
from lesview import *

In [ ]:
casename = 'lsc_ymc22_sbl_bbl_v2'
g = 9.81
N2 = 1.962e-4
T0 = 20.0
alphaT = 2.0e-4
H = 30
amplitude = 1 # m
wavelength = 60. # m
dTdz = N2/alphaT/g
dTdz

In [ ]:
datapath = os.path.join(os.path.pardir, 'oceananigans', '{:s}'.format(casename))
filepath = os.path.join(datapath, 'averages.jld2')
data_pfl = OceananigansDataProfile(filepath=filepath)

out_dir = os.path.join(os.path.pardir, 'data', 'gotm', casename)
os.makedirs(out_dir, exist_ok=True)

In [ ]:
ds = data_pfl.dataset
ds

In [ ]:
da = ds.data_vars['b']/alphaT/g + T0 - dTdz*H
darr = da.data.transpose()
time = pd.to_datetime(da.time.data)
z = da.z.data
dat_dump_pfl(time, z, [darr], os.path.join(out_dir, 't_prof.dat'), order=1)
da.plot()

In [ ]:
dax = ds.data_vars['u']
day = ds.data_vars['v']
darrx = copy.deepcopy(dax.data.transpose())
darry = copy.deepcopy(day.data.transpose())
time = pd.to_datetime(dax.time.data)
z = dax.z.data

dat_dump_pfl(time, z, [darrx, darry], os.path.join(out_dir, 'u_prof_l.dat'), order=1)
plt.figure()
dax.plot()
plt.figure()
day.plot()

In [ ]:
dax[:,0].plot(y='z', linestyle='--', color='gray', label='$u_i$ (ref)')
plt.plot(darrx[0,:], z, linestyle='--', color='k', label='$u_i$')
dax[:,-200:].mean(dim='time').plot(y='z', linestyle='-', color='gray', label='$\overline{u}$ (ref)')
plt.plot(np.mean(darrx[-200:,:], axis=0), z, linestyle='-', color='k', label='$\overline{u}$')
plt.legend()

In [ ]:
# Stokes drift
wavenumber = 2. * np.pi / wavelength # 1/m
frequency = np.sqrt(g * wavenumber * np.tanh(wavenumber * H)) # 1/s
us0 = amplitude**2 * wavenumber * frequency # m/s
us = us0 * np.cosh(2. * wavenumber * (z + H)) / (2. * np.sinh(wavenumber * H)**2)

In [ ]:
darrx = copy.deepcopy(dax.data.transpose())
darry = copy.deepcopy(day.data.transpose())
# remove Stokes drift -- Eulerian current
for i in np.arange(0,time.size):
    darrx[i,:] -= us

dat_dump_pfl(time, z, [darrx, darry], os.path.join(out_dir, 'u_prof_e.dat'), order=1)
plt.figure()
dax.plot()
plt.figure()
day.plot()

In [ ]:
dax[:,0].plot(y='z', linestyle='--', color='gray', label='$u_i$ (ref)')
plt.plot(darrx[0,:], z, linestyle='--', color='k', label='$u_i$')
dax[:,-200:].mean(dim='time').plot(y='z', linestyle='-', color='gray', label='$\overline{u}$ (ref)')
plt.plot(np.mean(darrx[-200:,:], axis=0), z, linestyle='-', color='k', label='$\overline{u}$')
plt.legend()

In [ ]:
darrx = copy.deepcopy(dax.data.transpose())
darry = copy.deepcopy(day.data.transpose())
# add additional Stokes drift -- Assuming u^E=U_0
for i in np.arange(0,time.size):
    darrx[i,:] += us

dat_dump_pfl(time, z, [darrx, darry], os.path.join(out_dir, 'u_prof_l2.dat'), order=1)
plt.figure()
dax.plot()
plt.figure()
day.plot()

In [ ]:
dax[:,0].plot(y='z', linestyle='--', color='gray', label='$u_i$ (ref)')
plt.plot(darrx[0,:], z, linestyle='--', color='k', label='$u_i$')
dax[:,-200:].mean(dim='time').plot(y='z', linestyle='-', color='gray', label='$\overline{u}$ (ref)')
plt.plot(np.mean(darrx[-200:,:], axis=0), z, linestyle='-', color='k', label='$\overline{u}$')
plt.legend()